# AI Presales Lab：llama.cpp 在 Google Colab 的 Q4/Q8 实测

这个 notebook 会在一次全新的 Colab runtime 中：

1. 固定 llama.cpp commit，并用官方 CUDA CMake 选项构建 `llama-server`。
2. 从 Qwen 官方 Hugging Face 仓库的固定 revision 下载 Q4_K_M 与 Q8_0。
3. 仅绑定到 `127.0.0.1`，预热后分别执行单并发和 4 并发压测。
4. 记录 Q4/Q8 的文件大小、延迟、TTFT、总生成吞吐、CPU RSS、GPU VRAM 和 JSON 输出通过率。
5. 将原始 JSON 报告和汇总报告写入 `data/results/colab/`。

注意：Colab GPU 类型、可用时长和配额是动态的；报告必须以本次 runtime 的输出为准，不能把本仓库示例或其他机器的结果写进简历。不要为 llama-server 开公网隧道，也不要放入客户数据。

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

# 第一次运行前只需要把这里改成你的公开 GitHub 仓库地址。
PROJECT_REPO_URL = 'PASTE_YOUR_PUBLIC_GITHUB_REPO_URL_HERE'
LLAMA_CPP_COMMIT = '6a1a922d269908a29cbd4b49c27e6a8e7fd10fae'
MODEL_REPO = 'Qwen/Qwen2.5-0.5B-Instruct-GGUF'
MODEL_REVISION = '2ed9be962c95f7625f4963ff51ed472e4538187a'
MODEL_FILES = {
    'Q4': 'qwen2.5-0.5b-instruct-q4_k_m.gguf',
    'Q8': 'qwen2.5-0.5b-instruct-q8_0.gguf',
}
CONTEXT = 2048
REQUESTS = 10
CONCURRENCIES = (1, 4)

if not PROJECT_REPO_URL.startswith('https://') or 'PASTE_' in PROJECT_REPO_URL:
    raise ValueError('请先把 PROJECT_REPO_URL 改成你的公开 GitHub 仓库地址。')

WORKDIR = Path('/content/ai-presales-colab')
REPO_DIR = WORKDIR / 'ai-presales-lab'
LLAMA_DIR = WORKDIR / 'llama.cpp'
MODEL_DIR = WORKDIR / 'models'
REPORT_DIR = REPO_DIR / 'data' / 'results' / 'colab'
WORKDIR.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(REPO_DIR)], check=True)
print('workspace:', WORKDIR)
print('project:', REPO_DIR)
print('model revision:', MODEL_REVISION)

In [ ]:
# Colab 免费 GPU 不保证型号；没有 NVIDIA GPU 时直接停止，避免生成不可比的 CPU 报告。
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('当前 runtime 没有 NVIDIA GPU，请在 Runtime > Change runtime type 中选择 GPU 后重连。')
subprocess.run(['nvidia-smi'], check=False)

if not LLAMA_DIR.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', 'https://github.com/ggml-org/llama.cpp.git', str(LLAMA_DIR)], check=True)
subprocess.run(['git', '-C', str(LLAMA_DIR), 'fetch', '--depth', '1', 'origin', LLAMA_CPP_COMMIT], check=True)
subprocess.run(['git', '-C', str(LLAMA_DIR), 'checkout', '--detach', LLAMA_CPP_COMMIT], check=True)

build_dir = LLAMA_DIR / 'build'
subprocess.run([
    'cmake', '-S', str(LLAMA_DIR), '-B', str(build_dir),
    '-DGGML_CUDA=ON', '-DGGML_NATIVE=OFF',
    '-DCMAKE_BUILD_TYPE=Release', '-DBUILD_SHARED_LIBS=OFF',
], check=True)
subprocess.run(['cmake', '--build', str(build_dir), '--config', 'Release', '--target', 'llama-server', 'llama-cli', '--parallel', '2'], check=True)
SERVER_BIN = build_dir / 'bin' / 'llama-server'
assert SERVER_BIN.exists(), SERVER_BIN
print(subprocess.check_output([str(SERVER_BIN), '--version'], text=True).strip())
subprocess.run([str(SERVER_BIN), '--list-devices'], check=False)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
from huggingface_hub import hf_hub_download

MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_paths = {}
for quantization, filename in MODEL_FILES.items():
    model_paths[quantization] = Path(hf_hub_download(
        repo_id=MODEL_REPO,
        filename=filename,
        revision=MODEL_REVISION,
        local_dir=str(MODEL_DIR),
    ))
    print(quantization, model_paths[quantization])

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

for quantization, path in model_paths.items():
    with path.open('rb') as handle:
        assert handle.read(4) == b'GGUF', f'not a GGUF file: {path}'
    print(json.dumps({
        'quantization': quantization,
        'filename': path.name,
        'size_mib': round(path.stat().st_size / (1024 * 1024), 2),
        'sha256': sha256(path),
    }, ensure_ascii=False))

os.environ.update({
    'LLAMA_CPP_COMMIT': LLAMA_CPP_COMMIT,
    'LLAMA_CPP_VERSION': subprocess.check_output([str(SERVER_BIN), '--version'], text=True).strip(),
    'MODEL_REPO': MODEL_REPO,
    'MODEL_REVISION': MODEL_REVISION,
    'LLAMA_BASE_URL': 'http://127.0.0.1:8080',
})
sys.path.insert(0, str(REPO_DIR / 'src'))

In [ ]:
from urllib.request import Request, urlopen

PORT = 8080
server_log_handle = None

def wait_for_health(process):
    for _ in range(120):
        if process.poll() is not None:
            break
        try:
            with urlopen(f'http://127.0.0.1:{PORT}/health', timeout=2) as response:
                if response.status == 200 and json.loads(response.read()).get('status') == 'ok':
                    return
        except Exception:
            time.sleep(1)
    log_path = WORKDIR / 'server.log'
    if log_path.exists():
        print(log_path.read_text(errors='replace')[-6000:])
    raise RuntimeError('llama-server 在 120 秒内没有健康起来。')

def start_server(model_path):
    global server_log_handle
    log_path = WORKDIR / 'server.log'
    server_log_handle = log_path.open('w', encoding='utf-8')
    process = subprocess.Popen([
        str(SERVER_BIN), '--model', str(model_path),
        '--ctx-size', str(CONTEXT), '--n-gpu-layers', '999',
        '--parallel', '4', '--seed', '42',
        '--host', '127.0.0.1', '--port', str(PORT),
    ], stdout=server_log_handle, stderr=subprocess.STDOUT)
    wait_for_health(process)
    return process

def stop_server(process):
    global server_log_handle
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    if server_log_handle is not None:
        server_log_handle.close()
        server_log_handle = None

def warmup():
    payload = json.dumps({
        'model': 'local-model',
        'messages': [{'role': 'user', 'content': 'Reply with only OK.'}],
        'temperature': 0, 'max_tokens': 8, 'stream': False,
    }).encode()
    request = Request('http://127.0.0.1:8080/v1/chat/completions', data=payload, headers={'Content-Type': 'application/json'})
    with urlopen(request, timeout=120) as response:
        assert response.status == 200
    print('warmup complete')

In [ ]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)
report_paths = []

for quantization, model_path in model_paths.items():
    server_process = start_server(model_path)
    try:
        for concurrency in CONCURRENCIES:
            warmup()
            output_path = REPORT_DIR / f'{quantization.lower()}-colab-c{concurrency}.json'
            command = [
                sys.executable, str(REPO_DIR / 'scripts' / 'benchmark_llama.py'),
                '--label', f'{quantization.lower()}-colab-c{concurrency}',
                '--requests', str(REQUESTS), '--concurrency', str(concurrency),
                '--model-path', str(model_path), '--context', str(CONTEXT),
                '--gpu-layers', '999', '--server-pid', str(server_process.pid),
                '--output', str(output_path),
            ]
            completed = subprocess.run(command, cwd=REPO_DIR, env=os.environ.copy())
            if completed.returncode != 0:
                raise RuntimeError(f'benchmark failed: {output_path}')
            report_paths.append(output_path)
    finally:
        stop_server(server_process)

print('raw reports:')
for path in report_paths:
    print(path)

In [ ]:
summary_path = REPORT_DIR / 'q4-q8-colab-summary.json'
subprocess.run([
    sys.executable, str(REPO_DIR / 'scripts' / 'summarize_benchmarks.py'),
    *[str(path) for path in report_paths], '--output', str(summary_path),
], cwd=REPO_DIR, check=True)

summary = json.loads(summary_path.read_text(encoding='utf-8'))
import pandas as pd
display(pd.DataFrame(summary['runs']))
print('summary:', summary_path)
print('请把本次实际输出的 JSON 报告和 runtime 硬件信息一起保存，再引用到简历或面试。')